# jepa-as-a-tool: the overnight run

Everything in one notebook: three worlds trained, a fourth evaluated only, one translator for all,
learning from use, and every agent evaluation.

| Stage | What | GPU | Time (est.) |
|---|---|---|---|
| 1. Data | tabletop + billiards (5,000 × 8 pushes each), new look (500), the agent's own PushT pushes (~16k) | CPU | ~20 min |
| 2. Evaluations without a new model | start in the background now: alone and with the real physics in every new world; PushT alone and with v2 on new tasks | CPU + API | run all night |
| 3. V-JEPA 2 latents | every dataset, spread over the two GPUs | 2 × T4 | ~40 min |
| 4. Translators | specialists, the shared one (3 worlds), the transfer test, **Gate 3** | 2 × T4 | ~1 h |
| 5. Predictors | tabletop ‖ billiards, then PushT fine-tuned on the agent's pushes; each world's evaluation with its model starts when it is trained | 2 × T4 | ~2.5 h |
| 6. Finish | wait for the evaluations, reports, `overnight.zip` | — | until the last episode |

About 1,350 episodes in contributor mode (~$12–15). Total ~8–9 h: inside Kaggle's 12 h.

**Before running (Kaggle):**
1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add-ons → Secrets → `META_API_KEY`, ticked for this notebook.
3. Add Input → Upload → a new dataset with **`overnight-inputs.zip`** (world model v2 + the 450 PushT
   evaluation logs; Kaggle may unzip it, both work).
4. Optional, saves ~1 h: add the output of the v2 training notebook as an input, if it still holds
   `data/random-12k` with `latents.npy`. Without it, PushT data is generated again (6,000 episodes).
5. **Save Version → Save & Run All**: it keeps running with the browser closed.

Every stage skips what is already done, so running a cell again continues where it stopped.

In [ ]:
import os

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    WORK = os.getcwd()
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .
!git log --oneline -1

pip's complaints about other packages are harmless. If it says it upgraded **numpy** or **torch**,
restart the kernel and run the cell above again.

## 0. Machine check: GPUs, disk, and MuJoCo rendering

The 3D worlds render with MuJoCo through EGL (the GPU). If EGL fails, the cell installs OSMesa
(software rendering, slower but works).

In [ ]:
import subprocess
import sys

!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h /tmp {WORK} | tail -2

CHECK = "from jat.worlds import get_world; f = get_world('billiards').render([150, 256, 400, 100]); print('render ok', f.shape, f.mean().round(1))"


def renders(gl):
    env = {**os.environ, "MUJOCO_GL": gl}
    out = subprocess.run([sys.executable, "-c", CHECK], env=env, capture_output=True, text=True)
    print(gl, "->", (out.stdout or out.stderr).strip().splitlines()[-1])
    return out.returncode == 0


if renders("egl"):
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq install -y libosmesa6 > /dev/null
    assert renders("osmesa"), "MuJoCo cannot render here"
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

## 1. API key and model

In [ ]:
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except ImportError:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name)
    except ImportError:
        pass
    from getpass import getpass

    return getpass(f"{name}: ")


os.environ["META_API_KEY"] = secret("META_API_KEY")
os.environ["META_API_BASE_URL"] = "https://api.meta.ai/v1"
# Contributor mode: same model, ~10x cheaper; Meta may train on these requests.
os.environ["META_MODEL"] = "muse-spark-1.3-contributor"
print("key loaded:", bool(os.environ["META_API_KEY"]), "| model:", os.environ["META_MODEL"])

## 2. Inputs

Finds `wm-v2/` and `eval-wm-v2/` (from `overnight-inputs.zip`, unzipped by Kaggle or here), and the
PushT v2 data (`random-12k/` with `latents.npy`) if attached.

In [ ]:
import glob
import zipfile
from pathlib import Path

INPUTS = Path(WORK) / "inputs"
ROOTS = ["/kaggle/input", str(INPUTS)]


def search(pattern):
    return sorted(Path(p) for r in ROOTS for p in glob.glob(f"{r}/**/{pattern}", recursive=True))


for archive in search("*.zip"):
    if archive.name.startswith(("overnight-inputs", "wm-v2", "eval-wm-v2")):
        zipfile.ZipFile(archive).extractall(INPUTS)

WM_V2 = next((p.parent for p in search("wm-v2/predictor.pt")), None)
EVAL_V2 = next((p.parent for p in search("eval-wm-v2/episodes")), None)
PUSHT_DATA = next((p.parent for p in search("random-12k/latents.npy")), None)

print("wm-v2:     ", WM_V2 or "MISSING")
print(
    "eval-wm-v2:",
    EVAL_V2,
    f"({len(os.listdir(EVAL_V2 / 'episodes'))} episodes)" if EVAL_V2 else "MISSING",
)
print("PushT data:", PUSHT_DATA or "not attached: it will be generated")
assert WM_V2 and EVAL_V2, "attach overnight-inputs.zip: see 'Before running', step 3"

## 3. The pipeline

Big data goes to `/tmp` (not saved, ~25 GB of room needed); runs, results and logs go to
`overnight/` in the output. Each step's full log is in `overnight/logs/<step>.log`.

In [ ]:
from jat.pipeline import Pipeline

OUT = Path(WORK) / "overnight"
OUT.mkdir(exist_ok=True)
Path("/tmp/jat-data").mkdir(exist_ok=True)
if not (OUT / "data").exists():
    (OUT / "data").symlink_to("/tmp/jat-data")  # latents stay out of the saved output

P = Pipeline(
    OUT,
    size="full",
    wm_v2=WM_V2,
    eval_v2=EVAL_V2,
    pusht_data=PUSHT_DATA,
    gpus=2,
    provider="meta",
    eval_workers=6,
)

## Stage 1: data (CPU, ~20 min; +~10 min if PushT is generated)

In [ ]:
P.stage_data()

## Stage 2: evaluations that need no new model (background)

They start now and keep going during the rest of the notebook. `overnight/logs/eval-*.log` shows progress.

In [ ]:
P.stage_baseline_evals()

## Stage 3: V-JEPA 2 latents (2 GPUs, ~40 min; +~40 min if PushT is generated)

In [ ]:
P.stage_encode()

## Stage 4: translators and Gate 3 (~1 h)

What to look at:
- **Gate 2**: each specialist's error on its own world. T worlds: under ~5 px and ~3°. Billiards: under ~5 px.
- **Gate 3**: the shared translator within 25% of each specialist. The agent in each new world reads
  with the shared translator if it passes, with the specialist if not.
- **New look**: the tabletop head on frames it never saw (zero-shot).
- **Transfer**: per number of billiards training episodes, a head on a frozen trunk vs from scratch.

In [ ]:
summary = P.stage_translators()
summary["transfer"]

## Stage 5: predictors (~2.5 h)

Tabletop and billiards train in parallel, one per GPU, then PushT is fine-tuned on the agent's own
pushes. Each world's evaluation with its world model starts as soon as the model is saved.
Logs: `overnight/logs/predictor-*.log`.

In [ ]:
P.stage_predictors()

## Stage 6: finish

Waits for every evaluation (failed episodes get one retry), writes `report.txt` per world, and packs
`overnight.zip`: runs, results and logs, no datasets.

In [ ]:
archive = P.stage_finish()
!ls -lh {archive}
for report in sorted(OUT.glob("results/*/report.txt")):
    print(f"\n===== {report.parent.name}\n{report.read_text()}")

## Send back

Download `overnight/overnight.zip` (Output tab) and send it.